# 🔬 Quantization Methods Workbench

Шаблон для изучения и реализации методов квантования — от базовых до SOTA.

**Уже реализовано:** Uniform Quantization, GPTQ  
**В процессе:** RTN, AWQ, SmoothQuant, QuIP, WaterSIC, TurboQuant

---

### Архитектура notebook'а

```
1. Общая инфраструктура (метрики, визуализация, данные)
2. Базовый класс квантователя
3. Слоты для каждого метода — пиши только логику
4. Единый бенчмарк для сравнения всех методов
```

## 0. Установка зависимостей

In [ ]:
# !pip install torch numpy matplotlib scipy tqdm

## 1. Инфраструктура: метрики, визуализация, тестовые данные

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from typing import Dict, Tuple, Optional, List
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
from scipy import linalg
import time
from tqdm import tqdm

torch.manual_seed(42)
np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Метрики качества квантования
# ═══════════════════════════════════════════════════════════

@dataclass
class QuantResult:
    """Результат квантования одного слоя/матрицы."""
    name: str
    W_orig: torch.Tensor       # оригинальные веса
    W_quant: torch.Tensor      # деквантованные веса (float)
    bits: float                 # среднее кол-во бит на элемент
    time_sec: float             # время квантования
    extra: Dict = field(default_factory=dict)  # доп. инфо (scale, zero_point, ...)


def mse(W: torch.Tensor, W_hat: torch.Tensor) -> float:
    """Mean Squared Error."""
    return ((W - W_hat) ** 2).mean().item()


def snr_db(W: torch.Tensor, W_hat: torch.Tensor) -> float:
    """Signal-to-Noise Ratio в dB."""
    signal = (W ** 2).mean().item()
    noise = ((W - W_hat) ** 2).mean().item()
    if noise < 1e-20:
        return float('inf')
    return 10 * np.log10(signal / noise)


def cos_sim(W: torch.Tensor, W_hat: torch.Tensor) -> float:
    """Cosine similarity (по всей матрице как вектор)."""
    w = W.flatten().float()
    wh = W_hat.flatten().float()
    return (torch.dot(w, wh) / (w.norm() * wh.norm() + 1e-12)).item()


def matmul_error(W: torch.Tensor, W_hat: torch.Tensor,
                 X: torch.Tensor) -> float:
    """Относительная ошибка матричного умножения: ||WX - W_hat X|| / ||WX||.
    
    Это ключевая метрика — именно её минимизируют GPTQ, WaterSIC и др.
    X — калибровочные данные (activations), shape: [d_in, n_samples].
    """
    WX = W.float() @ X.float()
    WX_hat = W_hat.float() @ X.float()
    return ((WX - WX_hat).norm() / (WX.norm() + 1e-12)).item()


def inner_product_bias(V: torch.Tensor, V_hat: torch.Tensor,
                       n_pairs: int = 1000) -> Tuple[float, float]:
    """Bias и variance оценки inner product (для TurboQuant).
    
    Берёт случайные пары строк, считает <v_i, v_j> vs <v_hat_i, v_hat_j>.
    Возвращает (mean_bias, std_error).
    """
    n = V.shape[0]
    idx_i = torch.randint(0, n, (n_pairs,))
    idx_j = torch.randint(0, n, (n_pairs,))
    
    true_ip = (V[idx_i] * V[idx_j]).sum(dim=-1)
    est_ip = (V_hat[idx_i] * V_hat[idx_j]).sum(dim=-1)
    
    errors = est_ip - true_ip
    return errors.mean().item(), errors.std().item()


def compute_all_metrics(result: QuantResult,
                        X: Optional[torch.Tensor] = None) -> Dict[str, float]:
    """Считает все метрики для QuantResult."""
    metrics = {
        "mse": mse(result.W_orig, result.W_quant),
        "snr_db": snr_db(result.W_orig, result.W_quant),
        "cos_sim": cos_sim(result.W_orig, result.W_quant),
        "bits": result.bits,
        "time_sec": result.time_sec,
    }
    if X is not None:
        metrics["matmul_rel_error"] = matmul_error(
            result.W_orig, result.W_quant, X
        )
    # Для KV-cache методов
    if result.extra.get("is_kv_cache", False):
        bias, std = inner_product_bias(result.W_orig, result.W_quant)
        metrics["ip_bias"] = bias
        metrics["ip_std"] = std
    return metrics

In [ ]:
# ═══════════════════════════════════════════════════════════
# Визуализация
# ═══════════════════════════════════════════════════════════

def plot_weight_distribution(results: List[QuantResult],
                             title: str = "Weight Distribution"):
    """Гистограммы распределения весов до/после квантования."""
    n = len(results)
    fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 3.5))
    
    # Оригинал
    orig = results[0].W_orig.flatten().cpu().numpy()
    axes[0].hist(orig, bins=100, alpha=0.7, color="steelblue", density=True)
    axes[0].set_title("Original (FP32)")
    axes[0].set_xlabel("value")
    
    for i, r in enumerate(results):
        q = r.W_quant.flatten().cpu().numpy()
        axes[i + 1].hist(q, bins=100, alpha=0.7, color="coral", density=True)
        axes[i + 1].set_title(f"{r.name} ({r.bits:.1f}b)")
        axes[i + 1].set_xlabel("value")
    
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


def plot_error_heatmap(results: List[QuantResult]):
    """Heatmap ошибки |W - W_hat| для каждого метода."""
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]
    
    for i, r in enumerate(results):
        err = (r.W_orig - r.W_quant).abs().cpu().numpy()
        im = axes[i].imshow(err, aspect="auto", cmap="hot")
        axes[i].set_title(f"{r.name} — |error|")
        plt.colorbar(im, ax=axes[i], fraction=0.046)
    
    plt.tight_layout()
    plt.show()


def plot_benchmark_comparison(all_metrics: Dict[str, Dict[str, float]]):
    """Сводная визуализация: bar chart по ключевым метрикам."""
    methods = list(all_metrics.keys())
    
    key_metrics = ["snr_db", "cos_sim", "mse", "time_sec"]
    available = [m for m in key_metrics if m in list(all_metrics.values())[0]]
    
    if "matmul_rel_error" in list(all_metrics.values())[0]:
        available.append("matmul_rel_error")
    
    fig, axes = plt.subplots(1, len(available), figsize=(4 * len(available), 4))
    if len(available) == 1:
        axes = [axes]
    
    colors = plt.cm.Set2(np.linspace(0, 1, len(methods)))
    
    for ax, metric in zip(axes, available):
        vals = [all_metrics[m].get(metric, 0) for m in methods]
        bars = ax.bar(methods, vals, color=colors)
        ax.set_title(metric)
        ax.tick_params(axis='x', rotation=45)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    f"{val:.4f}", ha='center', va='bottom', fontsize=8)
    
    fig.suptitle("Benchmark Comparison", fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_pareto(all_metrics: Dict[str, Dict[str, float]],
                x_metric: str = "bits", y_metric: str = "snr_db"):
    """Pareto chart: bits vs quality."""
    fig, ax = plt.subplots(figsize=(7, 5))
    for name, m in all_metrics.items():
        ax.scatter(m[x_metric], m[y_metric], s=100, zorder=3)
        ax.annotate(name, (m[x_metric], m[y_metric]),
                    textcoords="offset points", xytext=(5, 5), fontsize=9)
    ax.set_xlabel(x_metric)
    ax.set_ylabel(y_metric)
    ax.set_title(f"Pareto: {x_metric} vs {y_metric}")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# Генерация тестовых данных
# ═══════════════════════════════════════════════════════════

def make_test_weights(d_out: int = 256, d_in: int = 256,
                      style: str = "normal") -> torch.Tensor:
    """Генерирует тестовую матрицу весов.
    
    style:
      'normal'   — N(0, sqrt(2/d_in)) как Kaiming init
      'outliers' — normal + 1% outliers (имитация LLM)
      'llm_like' — колонки с разной дисперсией + outlier-каналы
    """
    if style == "normal":
        return torch.randn(d_out, d_in) * (2 / d_in) ** 0.5
    elif style == "outliers":
        W = torch.randn(d_out, d_in) * (2 / d_in) ** 0.5
        mask = torch.rand_like(W) < 0.01
        W[mask] *= 10
        return W
    elif style == "llm_like":
        # Разная дисперсия по каналам + несколько outlier-каналов
        scales = torch.rand(d_in) * 2 + 0.1
        W = torch.randn(d_out, d_in) * scales.unsqueeze(0)
        # 3 outlier-канала
        outlier_cols = torch.randint(0, d_in, (3,))
        W[:, outlier_cols] *= 8
        return W
    else:
        raise ValueError(f"Unknown style: {style}")


def make_calibration_data(d_in: int = 256, n_samples: int = 128,
                          style: str = "normal") -> torch.Tensor:
    """Калибровочные активации X, shape [d_in, n_samples]."""
    if style == "normal":
        return torch.randn(d_in, n_samples)
    elif style == "relu":
        return torch.randn(d_in, n_samples).clamp(min=0)
    elif style == "llm_like":
        X = torch.randn(d_in, n_samples)
        # Некоторые каналы активируются чаще
        mask = torch.rand(d_in, 1) > 0.3
        X = X * mask.float()
        return X
    else:
        raise ValueError(f"Unknown style: {style}")


def make_kv_cache_data(n_tokens: int = 512, d_head: int = 128,
                        n_heads: int = 8) -> torch.Tensor:
    """Имитация KV-cache: [n_heads, n_tokens, d_head]."""
    # Реалистичнее: нормируем по d_head + добавляем outlier-токены
    K = torch.randn(n_heads, n_tokens, d_head) / (d_head ** 0.5)
    # Несколько attention sink токенов с большой нормой
    K[:, :4, :] *= 5
    return K


# Создаём тестовые данные по умолчанию
W_test = make_test_weights(256, 256, "llm_like").to(DEVICE)
X_test = make_calibration_data(256, 128, "llm_like").to(DEVICE)
KV_test = make_kv_cache_data(512, 128, 8).to(DEVICE)

print(f"W_test:  {W_test.shape}, range [{W_test.min():.3f}, {W_test.max():.3f}]")
print(f"X_test:  {X_test.shape}")
print(f"KV_test: {KV_test.shape}")

## 2. Базовый класс квантователя

In [ ]:
class BaseQuantizer(ABC):
    """Базовый класс. Наследуйся и реализуй quantize()."""
    
    def __init__(self, bits: int = 4, **kwargs):
        self.bits = bits
        self.config = kwargs
    
    @abstractmethod
    def quantize(self, W: torch.Tensor,
                 X: Optional[torch.Tensor] = None,
                 **kwargs) -> Tuple[torch.Tensor, Dict]:
        """Квантует матрицу W.
        
        Args:
            W: [d_out, d_in] — матрица весов
            X: [d_in, n_samples] — калибровочные данные (опционально)
            
        Returns:
            W_quant: [d_out, d_in] — деквантованные веса (float)
            info: dict — доп. инфо (scale, zero_point, codebook, ...)
        """
        pass
    
    def run(self, W: torch.Tensor,
            X: Optional[torch.Tensor] = None,
            **kwargs) -> QuantResult:
        """Запуск квантования с замером времени."""
        t0 = time.time()
        W_quant, info = self.quantize(W, X, **kwargs)
        elapsed = time.time() - t0
        
        return QuantResult(
            name=self.__class__.__name__,
            W_orig=W.detach().cpu(),
            W_quant=W_quant.detach().cpu(),
            bits=info.get("effective_bits", self.bits),
            time_sec=elapsed,
            extra=info,
        )


class BaseKVQuantizer(ABC):
    """Базовый класс для KV-cache квантователей (TurboQuant и т.п.)."""
    
    def __init__(self, bits: float = 3.5, **kwargs):
        self.bits = bits
        self.config = kwargs
    
    @abstractmethod
    def quantize_vectors(self, V: torch.Tensor,
                         **kwargs) -> Tuple[torch.Tensor, Dict]:
        """Квантует набор векторов (ключи или значения).
        
        Args:
            V: [n_vectors, d] — матрица векторов
            
        Returns:
            V_quant: [n_vectors, d] — деквантованные векторы (float)
            info: dict
        """
        pass
    
    def run(self, V: torch.Tensor, **kwargs) -> QuantResult:
        """Запуск с замером."""
        t0 = time.time()
        V_quant, info = self.quantize_vectors(V, **kwargs)
        elapsed = time.time() - t0
        info["is_kv_cache"] = True
        
        return QuantResult(
            name=self.__class__.__name__,
            W_orig=V.detach().cpu(),
            W_quant=V_quant.detach().cpu(),
            bits=info.get("effective_bits", self.bits),
            time_sec=elapsed,
            extra=info,
        )

## 3. Утилиты, нужные нескольким методам

In [ ]:
# ═══════════════════════════════════════════════════════════
# Общие утилиты квантования
# ═══════════════════════════════════════════════════════════

def symmetric_quantize(x: torch.Tensor, bits: int
                       ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Симметричное скалярное квантование: x -> round(x/scale) -> clamp."""
    qmax = 2 ** (bits - 1) - 1
    scale = x.abs().max() / qmax
    scale = scale.clamp(min=1e-12)
    x_int = (x / scale).round().clamp(-qmax, qmax)
    return x_int * scale, scale


def asymmetric_quantize(x: torch.Tensor, bits: int
                        ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Асимметричное квантование с zero_point."""
    qmin, qmax = 0, 2 ** bits - 1
    x_min, x_max = x.min(), x.max()
    scale = (x_max - x_min) / (qmax - qmin)
    scale = scale.clamp(min=1e-12)
    zero_point = (qmin - x_min / scale).round()
    x_int = (x / scale + zero_point).round().clamp(qmin, qmax)
    x_deq = (x_int - zero_point) * scale
    return x_deq, scale, zero_point


def compute_hessian(X: torch.Tensor) -> torch.Tensor:
    """H = X @ X.T / n_samples — нужен для GPTQ, WaterSIC.
    
    X: [d_in, n_samples]
    Returns: [d_in, d_in]
    """
    n = X.shape[1]
    H = (X @ X.T) / n
    # Регуляризация для числовой стабильности
    damp = 0.01 * H.diagonal().mean()
    H += damp * torch.eye(H.shape[0], device=H.device)
    return H


def random_orthogonal_matrix(d: int, device: str = "cpu") -> torch.Tensor:
    """Случайная ортогональная матрица (Haar-distributed).
    
    Нужна для QuIP, TurboQuant (random rotation).
    """
    Z = torch.randn(d, d, device=device)
    Q, R = torch.linalg.qr(Z)
    # Гарантируем Haar distribution
    D = torch.diag(torch.sign(R.diagonal()))
    return Q @ D


def lloyd_max_quantizer(pdf_func, support: Tuple[float, float],
                        n_levels: int, n_iter: int = 50
                        ) -> Tuple[np.ndarray, np.ndarray]:
    """Lloyd-Max оптимальный скалярный квантователь.
    
    Нужен для TurboQuant — оптимальный квантователь для Beta-распределения.
    
    Returns:
        centroids: [n_levels] — значения реконструкции
        boundaries: [n_levels + 1] — границы решения
    """
    from scipy.integrate import quad
    
    a, b = support
    # Начальные центроиды — равномерно
    centroids = np.linspace(a, b, n_levels + 2)[1:-1]
    
    for _ in range(n_iter):
        # Boundaries: midpoints
        boundaries = np.concatenate([
            [a],
            (centroids[:-1] + centroids[1:]) / 2,
            [b]
        ])
        
        # Update centroids: E[x | boundaries]
        new_centroids = np.zeros_like(centroids)
        for i in range(n_levels):
            lo, hi = boundaries[i], boundaries[i + 1]
            num, _ = quad(lambda x: x * pdf_func(x), lo, hi)
            den, _ = quad(pdf_func, lo, hi)
            new_centroids[i] = num / (den + 1e-15)
        
        if np.allclose(centroids, new_centroids, atol=1e-10):
            break
        centroids = new_centroids
    
    boundaries = np.concatenate([
        [a], (centroids[:-1] + centroids[1:]) / 2, [b]
    ])
    return centroids, boundaries


print("Utilities loaded ✓")

---

## 4. Методы квантования — слоты для реализации

Каждый метод — отдельная ячейка. Пиши только логику в `quantize()`.

### Дорожная карта методов:

| # | Метод | Тип | Суть | Статус |
|---|-------|-----|------|--------|
| 1 | **Uniform (RTN)** | Weight-only | Round-to-nearest, baseline | ✅ Готов |
| 2 | **GPTQ** | Weight-only | Последовательное квант. с Hessian-коррекцией | ✅ Готов |
| 3 | **AWQ** | Weight-only | Salient channels × scale | 🔲 TODO |
| 4 | **SmoothQuant** | W+A | Сглаживание outliers из активаций в веса | 🔲 TODO |
| 5 | **QuIP** | Weight-only | Random rotation + incoherence processing | 🔲 TODO |
| 6 | **WaterSIC** | Weight-only | GPTQ + waterfilling (неравн. распр. бит) | 🔲 TODO |
| 7 | **TurboQuant** | KV-cache | PolarQuant + 1-bit QJL residual | 🔲 TODO |

### 4.1 Uniform / RTN (Round-to-Nearest) — BASELINE

In [ ]:
class UniformQuantizer(BaseQuantizer):
    """Round-to-nearest с per-channel symmetric квантованием.
    Самый простой baseline.
    """
    
    def quantize(self, W, X=None, **kwargs):
        # ── ТВОЯ РЕАЛИЗАЦИЯ (или используй готовую) ──
        qmax = 2 ** (self.bits - 1) - 1
        # Per-channel scale
        scale = W.abs().amax(dim=1, keepdim=True) / qmax
        scale = scale.clamp(min=1e-12)
        W_int = (W / scale).round().clamp(-qmax, qmax)
        W_deq = W_int * scale
        # ── КОНЕЦ ──
        return W_deq, {"scale": scale, "effective_bits": self.bits}

### 4.2 GPTQ

In [ ]:
class GPTQQuantizer(BaseQuantizer):
    """GPTQ: последовательное квантование с коррекцией через Hessian.
    
    Ключевая идея: квантуем столбец i, компенсируем ошибку 
    в оставшихся столбцах через H^{-1}.
    
    Ref: Frantar et al., "GPTQ: Accurate Post-Training Quantization 
         for Generative Pre-trained Transformers", ICLR 2023.
    """
    
    def quantize(self, W, X=None, **kwargs):
        assert X is not None, "GPTQ требует калибровочные данные X"
        
        # ── ТВОЯ РЕАЛИЗАЦИЯ ──
        # Подсказка по алгоритму:
        # 1. H = compute_hessian(X)
        # 2. H_inv = torch.linalg.cholesky(H) → solve
        # 3. Для каждого столбца i (или блоками):
        #    a. q_i = quantize(W[:, i])
        #    b. err_i = W[:, i] - q_i
        #    c. W[:, i+1:] += err_i * H_inv[i, i+1:] / H_inv[i, i]
        
        raise NotImplementedError("Вставь свою реализацию GPTQ")
        # ── КОНЕЦ ──
        
        # return W_deq, {"effective_bits": self.bits, "hessian_diag": ...}

### 4.3 AWQ (Activation-Aware Weight Quantization)

In [ ]:
class AWQQuantizer(BaseQuantizer):
    """AWQ: защита salient каналов через масштабирование.
    
    Ключевая идея: ~1% весов «важны» (определяется через активации).
    Вместо mixed-precision, масштабируем важные каналы вверх 
    перед квантованием → меньше relative error на них.
    
    Ref: Lin et al., "AWQ: Activation-aware Weight Quantization 
         for LLM Compression and Acceleration", MLSys 2024.
    """
    
    def quantize(self, W, X=None, **kwargs):
        assert X is not None, "AWQ требует калибровочные данные X"
        
        # ── ТВОЯ РЕАЛИЗАЦИЯ ──
        # Подсказка:
        # 1. Найти salient каналы: s_j = mean(|X[j, :]|)
        # 2. Поиск оптимального scale α по сетке:
        #    for alpha in [0, 0.1, ..., 1.0]:
        #        channel_scale = s^alpha
        #        W_scaled = W * channel_scale
        #        W_q = quantize(W_scaled)
        #        W_deq = W_q / channel_scale
        #        loss = ||W @ X - W_deq @ X||
        # 3. Применить лучший alpha
        
        raise NotImplementedError("Вставь свою реализацию AWQ")
        # ── КОНЕЦ ──

### 4.4 SmoothQuant (W8A8)

In [ ]:
class SmoothQuantQuantizer(BaseQuantizer):
    """SmoothQuant: перенос сложности квантования с активаций на веса.
    
    Ключевая идея: Y = (X · diag(s)^{-1}) · (diag(s) · W)
    Выбрать s так, чтобы и X_smooth и W_smooth были легко квантуемы.
    s_j = max(|X_j|)^α / max(|W_j|)^(1-α), α обычно 0.5
    
    Ref: Xiao et al., "SmoothQuant: Accurate and Efficient PTQ 
         for Large Language Models", ICML 2023.
    """
    
    def __init__(self, bits: int = 8, alpha: float = 0.5, **kwargs):
        super().__init__(bits, **kwargs)
        self.alpha = alpha
    
    def quantize(self, W, X=None, **kwargs):
        assert X is not None, "SmoothQuant требует активации"
        
        # ── ТВОЯ РЕАЛИЗАЦИЯ ──
        # Подсказка:
        # 1. act_scale = X.abs().amax(dim=1)  # [d_in]
        # 2. w_scale = W.abs().amax(dim=0)    # [d_in]
        # 3. s = act_scale^α / w_scale^(1-α)
        # 4. W_smooth = W * s.unsqueeze(0)
        # 5. X_smooth = X / s.unsqueeze(1)
        # 6. Квантуем W_smooth и X_smooth по отдельности
        # 7. Деквант: Y_hat = quant(X_smooth) @ quant(W_smooth).T
        #    → но для сравнения возвращаем W_smooth_dequant / s
        
        raise NotImplementedError("Вставь свою реализацию SmoothQuant")
        # ── КОНЕЦ ──

### 4.5 QuIP (Quantization with Incoherence Processing)

In [ ]:
class QuIPQuantizer(BaseQuantizer):
    """QuIP: Random rotation + LDLQ квантование.
    
    Ключевая идея: применить случайное ортогональное преобразование
    к весам и Hessian, чтобы сделать их «incoherent» — 
    все элементы примерно одного масштаба → uniform quantization работает лучше.
    
    Ref: Chee et al., "QuIP: 2-Bit Quantization of LLMs 
         With Guarantees", NeurIPS 2023.
    """
    
    def quantize(self, W, X=None, **kwargs):
        assert X is not None, "QuIP требует калибровочные данные"
        
        # ── ТВОЯ РЕАЛИЗАЦИЯ ──
        # Подсказка:
        # 1. U = random_orthogonal_matrix(d_out)
        # 2. V = random_orthogonal_matrix(d_in)
        # 3. W_rot = U @ W @ V  (incoherent)
        # 4. H_rot = V.T @ H @ V
        # 5. Применить GPTQ/LDLQ к W_rot с H_rot
        # 6. W_deq = U.T @ W_rot_quant @ V.T
        
        raise NotImplementedError("Вставь свою реализацию QuIP")
        # ── КОНЕЦ ──

### 4.6 WaterSIC ⭐

**Теория:** WaterSIC — вариант GPTQ/LDLQ, использующий waterfilling для неравномерного распределения бит по координатам. Стандартный GPTQ выделяет одинаковое число бит каждому весу. WaterSIC аллоцирует больше бит каналам с большей дисперсией (как в water-filling для channel capacity).

**Ключевой результат:** WaterSIC использует только скалярные INT-квантователи, но его MSE зависит только от det(Σ_X), а не от выбора базиса → иммунен к random rotations.

**Ref:** arXiv:2601.17187, Jan 2026.

In [ ]:
class WaterSICQuantizer(BaseQuantizer):
    """WaterSIC: GPTQ + waterfilling bit allocation.
    
    Отличие от GPTQ:
    - GPTQ: каждый столбец → одинаковое кол-во бит
    - WaterSIC: бит на столбец j пропорционален log(H_{jj})
      (waterfilling: больше бит → каналы с большей Hessian-дисперсией)
    
    Алгоритм (high-level):
    1. H = X @ X.T / n   (Hessian proxy)
    2. Waterfilling: R_j = R_avg + 0.5 * log2(H_jj / (prod H_jj)^{1/d})
       (R_j — кол-во бит для столбца j)
    3. SIC (Successive Interference Cancellation):
       Квантуем в порядке столбцов, аналогично GPTQ,
       но с R_j бит на столбец j.
    4. Ошибка от столбца j компенсируется в j+1..d через H^{-1}.
    
    Fundamental limit: D* = (1/d) * det(Σ_X)^{1/d} * 2^{-2R}
    WaterSIC достигает 2πe/12 * D* (≈ 0.25 bit от предела).
    """
    
    def quantize(self, W, X=None, **kwargs):
        assert X is not None, "WaterSIC требует калибровочные данные"
        
        # ── ТВОЯ РЕАЛИЗАЦИЯ ──
        # Подсказка по шагам:
        #
        # === ШАГ 1: Hessian и его Cholesky ===
        # H = compute_hessian(X)    # [d, d]
        # L = torch.linalg.cholesky(H)  # H = L @ L.T
        #
        # === ШАГ 2: Waterfilling bit allocation ===
        # diag_H = H.diagonal()
        # log_diag = torch.log2(diag_H)
        # R_avg = self.bits  # средний бюджет бит
        # # Waterfilling: R_j = R_avg + 0.5 * (log2(H_jj) - mean(log2(H_jj)))
        # R_per_col = R_avg + 0.5 * (log_diag - log_diag.mean())
        # R_per_col = R_per_col.clamp(min=2, max=8)  # clamp разумно
        # # Нормировка, чтобы средний = R_avg
        # R_per_col = R_per_col * (R_avg * d) / R_per_col.sum()
        #
        # === ШАГ 3: SIC — последовательное квантование ===
        # W_q = W.clone()
        # for j in range(d_in):
        #     # Квантуем столбец j с R_per_col[j] бит
        #     n_levels = int(2 ** R_per_col[j].item())
        #     q_j = quantize_column(W_q[:, j], n_levels)
        #     err_j = W_q[:, j] - q_j
        #     W_q[:, j] = q_j
        #     # Компенсация ошибки в оставшиеся столбцы
        #     if j < d_in - 1:
        #         W_q[:, j+1:] += err_j.unsqueeze(1) * (...H_inv correction...)
        
        raise NotImplementedError("Вставь свою реализацию WaterSIC")
        # ── КОНЕЦ ──
        
        # return W_deq, {
        #     "effective_bits": R_per_col.mean().item(),
        #     "bits_per_col": R_per_col.cpu(),
        #     "hessian_det": torch.linalg.det(H).item(),
        # }

### 4.7 TurboQuant ⭐ (KV-cache)

**Теория:** TurboQuant — двухстадийный онлайн векторный квантователь:

1. **PolarQuant (Stage 1):** Случайный поворот вектора → координаты следуют Beta-распределению → применяем оптимальный Lloyd-Max скалярный квантователь к каждой координате. Это даёт оптимальный MSE.

2. **QJL Residual (Stage 2):** MSE-оптимальный квантователь вносит bias в оценку inner product. 1-bit Quantized Johnson-Lindenstrauss transform на остаток убирает этот bias.

**Ключевой результат:** 3.5 bit → quality-neutral; 2.5 bit → marginal degradation. До 8x ускорение attention logits на H100.

**Ref:** Zandieh et al., arXiv:2504.19874, ICLR 2026.

In [ ]:
class TurboQuantQuantizer(BaseKVQuantizer):
    """TurboQuant: PolarQuant + 1-bit QJL residual.
    
    Для KV-cache квантования (ключи и значения в attention).
    
    Алгоритм:
    1. Генерируем случайную ортогональную матрицу R (один раз)
    2. Для каждого вектора v:
       a. v_rot = R @ v  (random rotation)
       b. Нормируем: norm = ||v_rot||, u = v_rot / norm
       c. Каждая координата u_i ~ Beta(1/2, (d-1)/2)
       d. Применяем Lloyd-Max квантователь к каждой u_i
       e. Восстанавливаем: v_hat_rot = dequant(u) * norm
       f. Residual: r = v_rot - v_hat_rot
       g. 1-bit QJL: sign(S @ r), где S — random sign matrix
       h. v_hat = R.T @ (v_hat_rot + qjl_correction)
    """
    
    def __init__(self, bits: float = 3.5, use_qjl: bool = True, **kwargs):
        super().__init__(bits, **kwargs)
        self.use_qjl = use_qjl
        self._codebooks = {}  # cache Lloyd-Max codebooks
    
    def _get_codebook(self, d: int, bits_per_coord: int):
        """Получить/кэшировать Lloyd-Max codebook для Beta distribution."""
        key = (d, bits_per_coord)
        if key not in self._codebooks:
            from scipy.stats import beta as beta_dist
            # После rotation, каждая координата единичного вектора
            # имеет распределение ~ Beta(1/2, (d-1)/2) (по модулю знака)
            a_param, b_param = 0.5, (d - 1) / 2
            pdf = lambda x: beta_dist.pdf(x, a_param, b_param)
            n_levels = 2 ** bits_per_coord
            centroids, boundaries = lloyd_max_quantizer(
                pdf, (0.0, 1.0), n_levels
            )
            self._codebooks[key] = (
                torch.tensor(centroids, dtype=torch.float32),
                torch.tensor(boundaries, dtype=torch.float32),
            )
        return self._codebooks[key]
    
    def quantize_vectors(self, V, **kwargs):
        # V: [n_vectors, d]
        
        # ── ТВОЯ РЕАЛИЗАЦИЯ ──
        # Подсказка:
        #
        # === SETUP ===
        # d = V.shape[-1]
        # R = random_orthogonal_matrix(d, device=V.device)  # [d, d]
        # centroids, boundaries = self._get_codebook(d, int(self.bits))
        #
        # === STAGE 1: PolarQuant ===
        # V_rot = V @ R.T               # random rotation
        # norms = V_rot.norm(dim=-1, keepdim=True)
        # U = V_rot / (norms + 1e-12)   # unit vectors
        # signs = U.sign()              # сохраняем знаки
        # U_abs = U.abs()               # квантуем |u_i|
        #
        # # Scalar quantization каждой координаты
        # U_q = torch.zeros_like(U_abs)
        # for каждого U_abs[i,j]:
        #     найти ближайший centroid → U_q[i,j]
        # (или vectorized: torch.bucketize + index)
        #
        # V_hat_rot = signs * U_q * norms
        #
        # === STAGE 2: QJL (optional) ===
        # if self.use_qjl:
        #     residual = V_rot - V_hat_rot
        #     # 1-bit QJL: random projection + sign
        #     # S = random Rademacher matrix [m, d], m << d
        #     # sketch = sign(S @ residual.T)
        #     # correction = (1/m) * S.T @ sketch (деквант)
        #     # V_hat_rot += correction_scaled
        #
        # === INVERSE ROTATION ===
        # V_hat = V_hat_rot @ R
        
        raise NotImplementedError("Вставь свою реализацию TurboQuant")
        # ── КОНЕЦ ──
        
        # return V_hat, {
        #     "effective_bits": actual_bits,
        #     "rotation_matrix": R.cpu(),
        #     "is_kv_cache": True,
        # }

### 4.8 Свободный слот — твой метод

In [ ]:
class CustomQuantizer(BaseQuantizer):
    """Место для твоего метода или комбинации."""
    
    def quantize(self, W, X=None, **kwargs):
        raise NotImplementedError("Implement me!")

---

## 5. Единый бенчмарк

In [ ]:
def run_weight_benchmark(
    quantizers: List[BaseQuantizer],
    W: torch.Tensor,
    X: torch.Tensor,
    title: str = "Weight Quantization Benchmark",
):
    """Запускает все weight-only квантователи и сравнивает."""
    results = []
    all_metrics = {}
    
    for q in quantizers:
        name = q.__class__.__name__
        try:
            result = q.run(W, X)
            metrics = compute_all_metrics(result, X)
            results.append(result)
            all_metrics[f"{name}({result.bits:.1f}b)"] = metrics
            print(f"✓ {name:25s} | {result.bits:.1f}b | "
                  f"SNR={metrics['snr_db']:.1f}dB | "
                  f"cos={metrics['cos_sim']:.6f} | "
                  f"MatMul err={metrics.get('matmul_rel_error', 'N/A')} | "
                  f"{result.time_sec:.3f}s")
        except NotImplementedError:
            print(f"⬜ {name:25s} | NOT IMPLEMENTED")
        except Exception as e:
            print(f"✗ {name:25s} | ERROR: {e}")
    
    if results:
        print(f"\n{'='*60}")
        plot_weight_distribution(results, title)
        plot_error_heatmap(results)
        if len(all_metrics) > 1:
            plot_benchmark_comparison(all_metrics)
            plot_pareto(all_metrics)
    
    return results, all_metrics


def run_kv_benchmark(
    quantizers: List[BaseKVQuantizer],
    KV: torch.Tensor,
    title: str = "KV-Cache Quantization Benchmark",
):
    """Запускает KV-cache квантователи (TurboQuant и т.п.)."""
    results = []
    all_metrics = {}
    
    # Берём первую голову для демо
    V = KV[0]  # [n_tokens, d_head]
    
    for q in quantizers:
        name = q.__class__.__name__
        try:
            result = q.run(V)
            metrics = compute_all_metrics(result)
            results.append(result)
            all_metrics[f"{name}({result.bits:.1f}b)"] = metrics
            
            ip_str = ""
            if "ip_bias" in metrics:
                ip_str = (f"IP bias={metrics['ip_bias']:.6f} "
                         f"IP std={metrics['ip_std']:.6f}")
            
            print(f"✓ {name:25s} | {result.bits:.1f}b | "
                  f"SNR={metrics['snr_db']:.1f}dB | "
                  f"cos={metrics['cos_sim']:.6f} | "
                  f"{ip_str} | "
                  f"{result.time_sec:.3f}s")
        except NotImplementedError:
            print(f"⬜ {name:25s} | NOT IMPLEMENTED")
        except Exception as e:
            print(f"✗ {name:25s} | ERROR: {e}")
    
    if results:
        print(f"\n{'='*60}")
        plot_weight_distribution(results, title)
        if len(all_metrics) > 1:
            plot_benchmark_comparison(all_metrics)
    
    return results, all_metrics

## 6. Запуск!

In [ ]:
# ═══════════════════════════════════════════════════════════
# Weight-only методы
# ═══════════════════════════════════════════════════════════

weight_quantizers = [
    UniformQuantizer(bits=4),
    UniformQuantizer(bits=3),
    # GPTQQuantizer(bits=4),         # раскомментируй когда реализуешь
    # AWQQuantizer(bits=4),
    # SmoothQuantQuantizer(bits=8),
    # QuIPQuantizer(bits=4),
    # WaterSICQuantizer(bits=4),
]

print("Weight-Only Quantization")
print("=" * 60)
w_results, w_metrics = run_weight_benchmark(
    weight_quantizers, W_test, X_test
)

In [ ]:
# ═══════════════════════════════════════════════════════════
# KV-Cache методы
# ═══════════════════════════════════════════════════════════

kv_quantizers = [
    # TurboQuantQuantizer(bits=4),     # раскомментируй
    # TurboQuantQuantizer(bits=3),
    # TurboQuantQuantizer(bits=2, use_qjl=True),
]

if kv_quantizers:
    print("\nKV-Cache Quantization")
    print("=" * 60)
    kv_results, kv_metrics = run_kv_benchmark(
        kv_quantizers, KV_test
    )

---

## 7. Шпаргалка: что квантуем и зачем

```
┌─────────────────────────────────────────────────────────────┐
│                  ЧТО КВАНТУЕМ В LLM                        │
├─────────────────┬───────────────────────────────────────────┤
│ Weights (W)     │ Статические, можно квантовать offline     │
│                 │ → GPTQ, AWQ, QuIP, WaterSIC              │
├─────────────────┼───────────────────────────────────────────┤
│ Activations (X) │ Динамические, outliers-проблема           │
│                 │ → SmoothQuant (W8A8)                      │
├─────────────────┼───────────────────────────────────────────┤
│ KV-Cache        │ Растёт с длиной контекста, online         │
│                 │ → TurboQuant, KIVI, KVQuant               │
└─────────────────┴───────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│              ЭВОЛЮЦИЯ ИДЕЙ                                  │
├─────────────────────────────────────────────────────────────┤
│ RTN (uniform)                                               │
│  └→ GPTQ (Hessian-коррекция ошибки, sequential)            │
│      ├→ AWQ (salient channels через активации)              │
│      ├→ QuIP (random rotation → incoherence)                │
│      └→ WaterSIC (waterfilling bit allocation, ≈optimal)    │
│                                                             │
│ SmoothQuant (W+A: перенос outliers W↔A)                    │
│                                                             │
│ KV-Cache:                                                   │
│  KIVI (asymmetric 2-bit)                                    │
│  └→ TurboQuant (PolarQuant + QJL, near-optimal)            │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│         КЛЮЧЕВЫЕ ФОРМУЛЫ                                    │
├─────────────────────────────────────────────────────────────┤
│ GPTQ row update:                                            │
│   q_j = quant(w_j)                                          │
│   δ_j = (w_j - q_j) / [H^{-1}]_{jj}                       │
│   w_{j+1:} += δ_j · [H^{-1}]_{j,j+1:}                     │
│                                                             │
│ WaterSIC bit allocation:                                    │
│   R_j = R_avg + ½ log₂(H_jj / (∏H_kk)^{1/d})             │
│   D* = (1/d)·det(Σ_X)^{1/d}·2^{-2R}   (fund. limit)      │
│                                                             │
│ TurboQuant PolarQuant:                                      │
│   v_rot = R·v  (random rotation)                            │
│   u_i ~ Beta(½, (d-1)/2)  after normalization               │
│   Lloyd-Max quantize each |u_i|                             │
│                                                             │
│ TurboQuant QJL residual (1-bit):                            │
│   sketch = sign(S · residual), S ∈ {±1}^{m×d}              │
│   → removes inner product estimation bias                   │
└─────────────────────────────────────────────────────────────┘
```